# ជំពូក ២៖ ការធ្វើការជាមួយទិន្នន័យអត្ថបទ (Working with Text Data)

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀមបរិស្ថាន
- ## ១. ការផ្ទុកអត្ថបទដើម
- ## ២. គំនិតបំបែកអត្ថបទជា token
- ## ៣. Preprocessing ពេញលេញជាង
- ## ៤. ការសាងសង់ vocabulary
- ## ៥. SimpleTokenizerV1
- ## ៦. ការពិនិត្យ vocabulary
- ## ៧. SimpleTokenizerV2 និង special tokens
- ## ៨. ប្រើ tiktoken (BPE ផលិតការ)
- ## ៩. គំនិត sliding window
- ## ១០. PyTorch DataLoader
- ## ១១. Embedding layer
- ## ១២. Positional embedding

## ០. ការត្រៀមបរិស្ថាន

ពិនិត្យកំណែបណ្ណាល័យ (torch, tiktoken ជាដើម) ដើម្បីធានាថាបរិស្ថានរបស់អ្នកត្រូវនឹងកូដ។

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

## ១. ការផ្ទុកអត្ថបទដើម

ផ្ទុកអត្ថបទឧទាហរណ់ the-verdict.txt ទាំងមូលចូលអថេរ raw_text ដើម្បីប្រើពេញជំពូក។

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

## ២. គំនិតបំបែកអត្ថបទជា token

ប្រើ re.split ដើម្បីបំបែកអត្ថបទជាឯកតាតូចៗ ហើយសង្កេតថា token មានអ្វីខ្លះ។

In [ ]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

In [ ]:
result = re.split(r'([,.]|\s)', text)

print(result)

In [ ]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

## ៣. Preprocessing ពេញលេញជាង

បំបែកលើសញ្ញាវណ្ណយុត្តបន្ថែម ហើយច្រោះចោលធាតុទទេ ដើម្បីបានបញ្ជី preprocessed ស្អាត។

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

In [ ]:
print(len(preprocessed))

## ៤. ការសាងសង់ vocabulary

បង្កើតសំណុំ token ទាំងអស់ (គ្មានស្ទួន) ហើយផ្គូផ្គាប់ token នីមួយៗទៅនឹងលេខគត់ (token ID)។

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

In [ ]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

## ៥. SimpleTokenizerV1

ក្លាស tokenizer ដំបូងដែលមាន encode (អត្ថបទ→IDs) និង decode (IDs→អត្ថបទ) ដោយផ្អែកលើ vocab។

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

In [ ]:
tokenizer.decode(ids)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## ៦. ការពិនិត្យ vocabulary

ពិនិត្យទំហំ vocabulary និង token ចុងក្រោយ ដើម្បីយល់ពីលំដាប់ផ្គូផ្គាប់។

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

## ៧. SimpleTokenizerV2 និង special tokens

កំណែកែលម្អដែលបន្ថែម special tokens ដូចជា <|unk|> និង <|endoftext|>។

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## ៨. ប្រើ tiktoken (BPE ផលិតការ)

ផ្ទុក encoding “gpt2” របស់បណ្ណាល័យ tiktoken ដែលជា BPE tokenizer ពិតប្រាកដ។

In [ ]:
# pip install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

In [ ]:
strings = tokenizer.decode(integers)

print(strings)

## ៩. គំនិត sliding window

យកផ្នែកបន្តបន្ទាប់នៃ token IDs មកបង្កើតជាគំរូបញ្ចូល/គោលដៅសម្រាប់ការបណ្តុះបណ្តាល។

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

## ១០. PyTorch DataLoader

សាងសង់ create_dataloader_v1 ដោយប្រើ Dataset និង DataLoader ជាមួយ batch_size និង stride។

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

## ១១. Embedding layer

nn.Embedding បំលែង token IDs ទៅជាវ៉ិចទ័រលេខ (embedding vectors)។

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
print(embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))

## ១២. Positional embedding

បូក token embeddings ជាមួយ positional embeddings ដើម្បីឱ្យម៉ូដែលដឹងលំដាប់ពាក្យ។

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ